# Example 8.1.3 / 1: Constraint Programming and Its MIP Reformulation

Companion notebook for **8.1.3 Constraint Programming and Complementarity Solvers** in the AMPL lab Documentation, section "Constraint Programming and Its MIP Reformulation". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Constraint Programming and Its MIP Reformulation

An MP-based MIP solver accepts the same logical operators by reformulating them. This model puts
the three steel products in three production slots, all different (`alldiff`), finishing the
long jobs late. HiGHS has no `alldiff`; its driver turns it into binary variables and linear
constraints. In `data.py` and `run.py`, `ampl` is an `AMPL` object that has already read
`model.mod`:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;                                    # products
param hours {PROD} > 0;                      # production hours of each product
var Slot {PROD} integer >= 1, <= 3;          # position of each product in the sequence
# hours weighted by position: long jobs placed early cost more
minimize Weighted_Slots: sum {p in PROD} hours[p] * Slot[p];
subject to Distinct: alldiff {p in PROD} Slot[p];   # each slot used once

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROD"] = ["bands", "coils", "plate"]
ampl.param["hours"] = {"bands": 30, "coils": 28.6, "plate": 21.9}

In [5]:
# run.py
ampl.solve(solver="highs", verbose=False)
print(ampl.solve_result, round(ampl.obj["Weighted_Slots"].value(), 1))
print(ampl.var["Slot"].to_dict())

solved 152.9
{'bands': 1, 'coils': 2, 'plate': 3}


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set PROD := bands coils plate;
param hours := bands 30  coils 28.6  plate 21.9;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
solve;
display Slot;

HiGHS 1.14.0: optimal solution; objective 152.9
11 simplex iterations
1 branching nodes
Slot [*] :=
bands  1
coils  2
plate  3
;

